### **Problem 4 :** Quantum Teleportation
The goal of quantum teleportation is to transfer the state of one qubit to another, using entanglement and classical communication.

Concept:
1. Entanglement: Create an entangled pair of qubits, A and B.
2. State Preparation: Prepare a third qubit, Q, in an arbitrary quantum state (the state you want to "teleport").
3. Measurement and Communication:
    - Measure Q and A in a specific basis (using CNOT and Hadamard gates).
    - Send the results of the measurement (classical bits) to the owner of B.

4. Reconstruction:
    - Based on the received classical bits, apply appropriate gates to 
B to recreate Q's state.


Circuit Overview :
We’ll create a 3-qubit circuit:
- Q: The qubit to be teleported.
- A: Entangled with 
- B: Receives Q's state after teleportation.

In [1]:
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit_aer import AerSimulator
from qiskit.compiler import transpile


In [2]:
# Create quantum and classical registers
q = QuantumRegister(3, 'q')  # 3 qubits
c = ClassicalRegister(2, 'c')  # 2 classical bits

In [3]:
# Create quantum circuit with registers
qc = QuantumCircuit(q, c)

# Prepare the state to teleport (arbitrary state on Q)
theta = 3.14 / 4          # the phase we will add - kept in a variable so we can re-use it later
qc.h(q[0])                # Create a superposition state on Q
qc.p(theta, q[0])         # Add a phase for complexity


In [4]:
# Create entanglement between qubits A and B
qc.h(q[1])  # Hadamard gate on A
qc.cx(q[1], q[2])  # CNOT between A (control) and B (target)

# Entangle Q with A and measure
qc.cx(q[0], q[1])  # CNOT between Q (control) and A (target)
qc.h(q[0])  # Hadamard gate on Q


In [5]:
# Measure Q and A
qc.measure(q[0], c[0])  # Measure Q (stored in classical bit 0)
qc.measure(q[1], c[1])  # Measure A (stored in classical bit 1)

#### The Correction Step : which bit controls which gate ?

After the measurement we hold two classical bits :

- `c[0]` = the result of measuring **Q** (the qubit whose state we are sending)
- `c[1]` = the result of measuring **A** (Alice's half of the entangled pair)

Working through the algebra of the four possible outcomes gives a simple rule :

| c[1] (A) | c[0] (Q) | Gate to apply on B |
|---|---|---|
| 0 | 0 | nothing |
| 1 | 0 | X |
| 0 | 1 | Z |
| 1 | 1 | X then Z |

So **X is controlled by A's bit** and **Z is controlled by Q's bit** - each one independently.

> ⚠️ A very common mistake is writing a condition on the *whole* classical register
> (e.g. "apply X if the register equals 1"). That compares both bits at once, so the
> `11` outcome would silently skip the X correction and the teleportation would fail
> a quarter of the time. Always condition on the individual bit.

We express this with `if_test`, the modern Qiskit way of writing a classically
controlled operation (the old `.c_if()` method was removed in Qiskit 2.0).

In [6]:
# Quantum teleportation correction, conditioned on the individual classical bits
with qc.if_test((c[1], 1)):   # A measured 1  ->  undo the bit flip
    qc.x(q[2])

with qc.if_test((c[0], 1)):   # Q measured 1  ->  undo the phase flip
    qc.z(q[2])


In [7]:
# Simulator
# NOTE: we use AerSimulator() here instead of Aer.get_backend('qasm_simulator').
# The old qasm_simulator backend cannot run classically controlled ("dynamic") circuits,
# and teleportation needs exactly that.
simulator = AerSimulator()


In [8]:
# Transpile and run the circuit
transpiled_circuit = transpile(qc, simulator)
job = simulator.run(transpiled_circuit, shots=1000)
result = job.result()


In [9]:
# Get results
counts = result.get_counts(qc)
print("Measurement Results:", counts)

Measurement Results: {'10': 235, '11': 252, '00': 277, '01': 236}


In [10]:
# Circuit visualization
qc.draw()

┌───┐┌──────────┐     ┌───┐┌─┐                                          »
q_0: ┤ H ├┤ P(0.785) ├──■──┤ H ├┤M├──────────────────────────────────────────»
     ├───┤└──────────┘┌─┴─┐└┬─┬┘└╥┘                                          »
q_1: ┤ H ├─────■──────┤ X ├─┤M├──╫───────────────────────────────────────────»
     └───┘   ┌─┴─┐    └───┘ └╥┘  ║   ┌──────  ┌───┐ ───────┐   ┌──────  ┌───┐»
q_2: ────────┤ X ├───────────╫───╫───┤ If-0  ─┤ X ├  End-0 ├───┤ If-0  ─┤ Z ├»
             └───┘           ║   ║   └──╥───  └───┘ ───────┘   └──╥───  └───┘»
                             ║   ║ ┌────╨────┐               ┌────╨────┐     »
c: 2/════════════════════════╩═══╩═╡ c_1=0x1 ╞═══════════════╡ c_0=0x1 ╞═════»
                             1   0 └─────────┘               └─────────┘     »
«               
«q_0: ──────────
«               
«q_1: ──────────
«      ───────┐ 
«q_2:   End-0 ├─
«      ───────┘ 
«c: 2/══════════
«

----
### Did the teleportation actually work ?

Look carefully at the counts above : they are the results of measuring **Q and A**, and
they come out roughly 25% each no matter what state we teleported. Those bits are
*random by construction* - they tell us nothing about whether B ended up in the right state.

To really check, we use a trick. If B truly holds the state

∣ψ⟩ = U∣0⟩,  where U = P(θ)·H

then applying the **inverse** operation U† to B must send it back to ∣0⟩. So we run the whole
protocol again, add U† on B at the very end, and measure B. If teleportation worked, we must
get `0` **100% of the time**.

In [11]:
# Rebuild the same protocol, but this time also measure B after un-preparing it
q = QuantumRegister(3, 'q')
c = ClassicalRegister(2, 'c')      # Q and A measurements
b = ClassicalRegister(1, 'b')      # B measurement (our verification bit)
check = QuantumCircuit(q, c, b)

# 1. Prepare the state to teleport on Q
check.h(q[0])
check.p(theta, q[0])

# 2. Entangle A and B
check.h(q[1])
check.cx(q[1], q[2])

# 3. Bell measurement on Q and A
check.cx(q[0], q[1])
check.h(q[0])
check.measure(q[0], c[0])
check.measure(q[1], c[1])

# 4. Corrections on B
with check.if_test((c[1], 1)):
    check.x(q[2])
with check.if_test((c[0], 1)):
    check.z(q[2])

# 5. Apply the INVERSE preparation on B and measure it
check.p(-theta, q[2])   # undo the phase
check.h(q[2])           # undo the Hadamard
check.measure(q[2], b[0])


In [12]:
result_check = simulator.run(transpile(check, simulator), shots=1000).result()
counts_check = result_check.get_counts()

# The counts key looks like "b c1c0" - we only care about the b bit
b_counts = {}
for bitstring, n in counts_check.items():
    b_bit = bitstring.split()[0]          # leftmost register is b
    b_counts[b_bit] = b_counts.get(b_bit, 0) + n

print("Full counts        :", counts_check)
print("B after un-preparing:", b_counts)
print()
success = b_counts.get('0', 0) / 1000
print(f"Teleportation success rate: {success * 100:.1f}%")


Full counts        : {'0 01': 238, '0 11': 248, '0 10': 270, '0 00': 244}
B after un-preparing: {'0': 1000}

Teleportation success rate: 100.0%


`100.0%` - every single shot returns `0`, which means B always ended up holding exactly the
state we prepared on Q. That is the real proof that the teleportation worked.

> Try it yourself : comment out one of the two correction blocks and re-run. The success rate
> drops to about 50% straight away, because half of the outcomes are left uncorrected.